# HHP Regression Figure Reproduction

This notebook reproduces the regression figure layer from deidentified archive datasets.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from pathlib import Path
DATA = Path('../data')
reg = pd.read_csv(DATA/'regression_analysis_dataset.csv')
early = pd.read_csv(DATA/'early_growth_dataset.csv')
reg.head()

## Figures S1-S4: Baseline diameter versus longitudinal growth

Panels are generated by segment and can be filtered to high-confidence series.

In [ ]:
for seg in ['ascending','sinus']:
    d = reg[(reg.segment==seg) & reg.best_slope_floored_mm_per_year.notna() & reg.baseline_diameter_cm.notna()]
    plt.figure(figsize=(6,4))
    plt.scatter(d['baseline_diameter_cm'], d['best_slope_floored_mm_per_year'], alpha=0.6)
    X = sm.add_constant(d['baseline_diameter_cm'])
    m = sm.OLS(d['best_slope_floored_mm_per_year'], X).fit()
    xs = np.linspace(d['baseline_diameter_cm'].min(), d['baseline_diameter_cm'].max(), 100)
    pred = m.get_prediction(sm.add_constant(xs)).summary_frame()
    plt.plot(xs, pred['mean'])
    plt.fill_between(xs, pred['mean_ci_lower'], pred['mean_ci_upper'], alpha=0.2)
    plt.xlabel('Baseline diameter (cm)')
    plt.ylabel('Floored growth rate (mm/year)')
    plt.title(f'{seg}: baseline diameter vs growth')
    plt.show()

## Age versus growth plots

Used for Figures S7-S12 with optional Top40 highlighting.

In [ ]:
for seg in ['ascending','sinus']:
    d = reg[(reg.segment==seg) & reg.best_slope_floored_mm_per_year.notna() & reg.age_first_hhp_years.notna()]
    plt.figure(figsize=(6,4))
    plt.scatter(d['age_first_hhp_years'], d['best_slope_floored_mm_per_year'], alpha=0.45, label='HHP')
    t = d[d.top40_member_mapped==1]
    plt.scatter(t['age_first_hhp_years'], t['best_slope_floored_mm_per_year'], alpha=0.9, label='Mapped Top40')
    X = sm.add_constant(d['age_first_hhp_years'])
    m = sm.OLS(d['best_slope_floored_mm_per_year'], X).fit()
    xs = np.linspace(d['age_first_hhp_years'].min(), d['age_first_hhp_years'].max(), 100)
    pred = m.get_prediction(sm.add_constant(xs)).summary_frame()
    plt.plot(xs, pred['mean'])
    plt.fill_between(xs, pred['mean_ci_lower'], pred['mean_ci_upper'], alpha=0.2)
    plt.xlabel('Age at first HHP echo (years)')
    plt.ylabel('Floored growth rate (mm/year)')
    plt.title(f'{seg}: age vs growth')
    plt.legend()
    plt.show()

## Figure S16: Early growth

Early growth is calculated from the first two available measurements per patient-segment.

In [ ]:
d = early.dropna(subset=['baseline_diameter_cm','early_rate_mm_per_year'])
plt.figure(figsize=(6,4))
plt.scatter(d['baseline_diameter_cm'], d['early_rate_mm_per_year'], alpha=0.5)
X = sm.add_constant(d['baseline_diameter_cm'])
m = sm.OLS(d['early_rate_mm_per_year'], X).fit()
xs = np.linspace(d['baseline_diameter_cm'].min(), d['baseline_diameter_cm'].max(), 100)
pred = m.get_prediction(sm.add_constant(xs)).summary_frame()
plt.plot(xs, pred['mean'])
plt.fill_between(xs, pred['mean_ci_lower'], pred['mean_ci_upper'], alpha=0.2)
plt.xlabel('Baseline diameter (cm)')
plt.ylabel('Early annualized growth (mm/year)')
plt.title('Baseline maximum diameter vs early annualized growth')
plt.show()
m.summary()

## Figures S18-S19: Delta versus AHI

The Top40 overlay is limited to records definitively mapped by exact last-name linkage.

In [ ]:
for subset, title in [(reg, 'HHP cohort'), (reg[reg.top40_member_mapped==1], 'Mapped Top40 subset')]:
    d = subset.dropna(subset=['delta_cm','AHI_cm_per_m'])
    plt.figure(figsize=(6,4))
    for seg in ['ascending','sinus']:
        g = d[d.segment==seg]
        plt.scatter(g['delta_cm'], g['AHI_cm_per_m'], alpha=0.7, label=seg)
    plt.xlabel('Observed - expected diameter (cm)')
    plt.ylabel('AHI (cm/m)')
    plt.title(title)
    plt.legend()
    plt.show()